# 📏 05 · Evaluation & Calibration

> **Chapter 5.** Bootstrap CIs on every headline metric, Murphy
> Brier decomposition, Platt vs isotonic calibration, operating
> points. This is where point estimates become defensible claims.

---

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path as _Path
sys.path.insert(0, str(_Path.cwd().parent))
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

REPO = Path.cwd().parent
boot = pd.read_csv(REPO / "results/metrics/xgboost_bootstrap_ci.csv")
print(boot.head().to_string(index=False))

## 1,000-replicate bootstrap CIs

Every number below has a 95 % nonparametric percentile interval from
1,000 bootstrap resamples of (y, p) pairs. Pairs where the resample
has a single class are silently skipped (extremely rare on test-set
sizes).

In [ ]:
rows = []
for _, r in boot.iterrows():
    for m in ["roc_auc", "pr_auc", "f1", "brier_loss", "mcc"]:
        rows.append({
            "slice": r["metric_set"],
            "metric": m,
            "mean": r[f"{m}__mean"],
            "ci_lo": r[f"{m}__ci_lo"],
            "ci_hi": r[f"{m}__ci_hi"],
        })
df = pd.DataFrame(rows)
test_df = df[df["slice"].str.startswith("test")]
print("Test split (1000-boot 95% CIs):")
print(test_df.to_string(index=False))

## Calibration deep dive (Stage 3)

### Brier decomposition

Murphy's additive identity:

> Brier = Reliability − Resolution + Uncertainty

* **Reliability** → miscalibration; fixable by Platt / isotonic.
* **Resolution** → discrimination; fixable only by a better model.
* **Uncertainty** → irreducible data entropy `ȳ(1 − ȳ)`.

In [ ]:
decomp = pd.read_csv(REPO / "results/metrics/brier_decomposition.csv")
test_decomp = decomp[decomp["split"] == "test"].copy()
print("Brier decomposition on test:")
cols = ["calibrator", "brier", "reliability", "resolution", "uncertainty", "ece", "log_loss"]
print(test_decomp[cols].to_string(index=False))

**Reading the table.** Resolution stays at 0.1053 across all three
calibrators — exactly as theory predicts, because monotone post-hoc
calibrators cannot change discrimination. Reliability drops 23× with
isotonic, pulling ECE from 0.054 to 0.011 (below the 0.02 target).

### Reliability triptych

The 3-panel plot below is `results/figures/calibration_triptych.png`
— rendered fresh so the notebook carries the figure inline.

In [ ]:
from IPython.display import Image  # noqa: PLC0415
Image(filename=str(REPO / "results/figures/calibration_triptych.png"))

## Operating points

For clinical use we often need "what threshold gets us recall ≥ 95 %?"
or "what precision is achievable at threshold X?" — the table below is
the canonical lookup.

In [ ]:
ops = pd.read_csv(REPO / "results/metrics/xgboost_operating_points.csv")
print(ops.to_string(index=False))

## Reproduce

```bash
python -m src.evaluate_baseline    # bootstrap + reliability + ops
python -m src.calibration_deep     # Brier decomposition + triptych
```